# Databricks SQL Dashboard: Pensjon Lakehouse

Denne notebooken inneholder de ferdige SQL-queryene for dashboardet,
og en steg-for-steg-guide til å sette det opp i Databricks UI.

**Forutsetning:** `01_bronze_ingest` og `02_silver_gold` er kjørt.

## Slik lager du dashboardet

1. Gå til **SQL → Dashboards → Create dashboard**
2. Gi det navnet **Pensjon Lakehouse**
3. Legg til panelene under, én for én
4. Hver seksjon viser: SQL-queryen du limer inn, visualiseringstype, og innstillinger

> **Tips:** Velg et SQL Warehouse som compute. Du trenger ikke en klynge for dashboards.

---

## Panel 1: KPI — Pensjonsandel (55+)

**Legg til:** Klikk `+` → **Add a visualization** → Lim inn queryen under

**Visualiseringstype:** Counter

**Innstillinger:**
- Value column: `pensjonsandel_pst`
- Counter label: `Pensjonsandel (55+)`
- Target column: *(la stå tom)*
- Row number: 1 (siste år)
- Suffix: ` %`
- Decimal places: 1

**Plassering:** Øverst til venstre, 1/3 bredde

In [0]:
-- Panel 1: KPI — Pensjonsandel
SELECT
    year,
    pensjonsandel_pst,
    total_55_pluss,
    total_befolkning
FROM pensjon_lakehouse.gold.pensjonsandel_trend
ORDER BY year DESC
LIMIT 1

---

## Panel 2: KPI — Totalt 55+

**Visualiseringstype:** Counter

**Innstillinger:**
- Value column: `total_55_pluss`
- Counter label: `Totalt 55+`
- Number format: `#,##0` (tusenskilletegn)

**Plassering:** Øverst i midten, 1/3 bredde

In [0]:
-- Panel 2: KPI — Totalt 55+
SELECT total_55_pluss, total_befolkning
FROM pensjon_lakehouse.gold.pensjonsandel_trend
ORDER BY year DESC
LIMIT 1

---

## Panel 3: KPI — Total befolkning

**Visualiseringstype:** Counter

**Innstillinger:**
- Value column: `total_befolkning`
- Counter label: `Total befolkning`
- Number format: `#,##0`

**Plassering:** Øverst til høyre, 1/3 bredde

In [0]:
-- Panel 3: KPI — Total befolkning
SELECT total_befolkning
FROM pensjon_lakehouse.gold.pensjonsandel_trend
ORDER BY year DESC
LIMIT 1

---

## Panel 4: Pensjonsandel-trend over tid

**Visualiseringstype:** Line

**Innstillinger:**
- X column: `year`
- Y columns: `pensjonsandel_pst`
- Y axis label: `Andel 55+ (%)`
- X axis label: `År`
- Show data points: ✅ Ja
- Y axis range: auto (eller sett min til 0)

**Plassering:** Venstre halvdel, rad 2, 1/2 bredde

In [0]:
-- Panel 4: Pensjonsandel-trend
SELECT
    CAST(year AS INT) AS year,
    pensjonsandel_pst
FROM pensjon_lakehouse.gold.pensjonsandel_trend
ORDER BY year

---

## Panel 5: Aldersfordeling siste år

**Visualiseringstype:** Bar

**Innstillinger:**
- X column: `aldersgruppe`
- Y columns: `befolkning`
- Color: fast farge eller `aldersgruppe` for fargekoding
- Data labels: ✅ Vis `andel_pst` som label (f.eks. «24.1 %»)
- Sort: etter `aldersgruppe_sortering` (automatisk fra ORDER BY)

**Plassering:** Høyre halvdel, rad 2, 1/2 bredde

In [0]:
-- Panel 5: Aldersfordeling siste år
SELECT
    aldersgruppe,
    aldersgruppe_sortering,
    befolkning,
    ROUND(andel * 100, 1) AS andel_pst
FROM pensjon_lakehouse.gold.aldersgruppe_fordeling
ORDER BY aldersgruppe_sortering

---

## Panel 6: Kommuner med høyest andel 55+

**Visualiseringstype:** Bar (horisontal)

**Innstillinger:**
- X column: `andel_pst`
- Y column: `kommune`
- Orientation: Horizontal
- Sort: etter `andel_pst` synkende (kommer fra ORDER BY)
- Data labels: ✅ Vis verdien (f.eks. «42.3 %»)
- Color: gradient fra lav til høy (valgfritt)

**Plassering:** Venstre halvdel, rad 3, 1/2 bredde

In [0]:
-- Panel 6: Top kommuner med høyest andel 55+
SELECT
    kommune_label AS kommune,
    total_befolkning AS innbyggere,
    pension_age_befolkning AS innbyggere_55_pluss,
    ROUND(pension_age_share * 100, 1) AS andel_pst
FROM pensjon_lakehouse.gold.top_kommuner_pensjonsalder
ORDER BY andel_pst DESC
LIMIT 10

---

## Panel 7: Næringer etter estimert pensjonsvolum

**Visualiseringstype:** Bar (horisontal)

**Innstillinger:**
- X column: `volum_mrd`
- Y column: `naering`
- Orientation: Horizontal
- Data labels: ✅ Vis verdien (f.eks. «12.4 mrd»)
- X axis label: `Est. pensjonsvolum (mrd kr)`

**Plassering:** Høyre halvdel, rad 3, 1/2 bredde

In [0]:
-- Panel 7: Næringer etter estimert pensjonsvolum
SELECT
    naering_label AS naering,
    lonsstakere,
    manedslonn,
    ROUND(estimert_pensjonsvolum / 1e9, 2) AS volum_mrd
FROM pensjon_lakehouse.gold.naering_pensjonsvolum
WHERE naering_label != 'Alle næringer'
ORDER BY volum_mrd DESC
LIMIT 10

---

## Panel 8: Aldersgruppe-trend over tid

**Visualiseringstype:** Area (stacked)

**Innstillinger:**
- X column: `year`
- Y columns: `andel_pst`
- Group by / Color: `aldersgruppe`
- Stacking: Stacked
- X axis label: `År`
- Y axis label: `Andel (%)`
- Legend: under grafen, horisontal

**Plassering:** Full bredde, rad 4

In [0]:
-- Panel 8: Aldersgruppe-trend over tid
SELECT
    CAST(year AS INT) AS year,
    aldersgruppe,
    ROUND(andel * 100, 1) AS andel_pst
FROM pensjon_lakehouse.gold.aldersgruppe_trend
ORDER BY year, aldersgruppe_sortering

---

## Panel 9: Detailtabell — alle kommuner

**Visualiseringstype:** Table

**Innstillinger:**
- Vis alle kolonner
- Sorterbar (klikk på kolonneheader)
- Conditional formatting på `andel_pst`: fargeskala fra grønn (lav) til rød (høy)
- Sider: 20 rader per side

**Plassering:** Full bredde, rad 5

In [0]:
-- Panel 9: Detailtabell — alle kommuner siste år
SELECT
    kommune_label AS kommune,
    total_befolkning AS innbyggere,
    pension_age_befolkning AS innbyggere_55_pluss,
    ROUND(pension_age_share * 100, 1) AS andel_pst
FROM pensjon_lakehouse.silver.befolkning_pensjon
WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_pensjon)
ORDER BY andel_pst DESC

---

## Dashboard-layout

```
┌──────────────────┬──────────────────┬──────────────────┐
│  Pensjonsandel   │   Totalt 55+     │ Total befolkning │
│     29.3 %       │   1 612 000      │    5 504 000     │
├──────────────────┴─────────┬────────┴──────────────────┤
│                            │                           │
│  Pensjonsandel-trend       │  Aldersfordeling          │
│  (linjegraf)               │  (søylediagram)           │
│                            │                           │
├────────────────────────────┼───────────────────────────┤
│                            │                           │
│  Top 10 kommuner           │  Top 10 næringer          │
│  (horisontale søyler)      │  (horisontale søyler)     │
│                            │                           │
├────────────────────────────┴───────────────────────────┤
│                                                        │
│  Aldersgruppe-trend over tid (stacked area)            │
│                                                        │
├────────────────────────────────────────────────────────┤
│                                                        │
│  Detailtabell — alle kommuner                          │
│                                                        │
└────────────────────────────────────────────────────────┘
```

## Tips for et pent dashboard

- Bruk **mørkt tema** i Dashboard Settings for en moderne look
- Sett lik høyde på paneler i samme rad
- Hold KPI-ene kompakte (lav høyde)
- Bruk konsistente farger: én farge for trend, fargeskala for rangeringer
- Legg til en **tekst-widget** øverst med tittel og datakilde:
  `Pensjon Lakehouse · SSB 07459 + 11654 · Bronze → Silver → Gold`